# QIAS 2026 RAG System with Qwen3.5 (HuggingFace)

Islamic Inheritance Reasoning with Retrieval Augmented Generation.
Uses Qwen3.5-9B via HuggingFace Transformers with native thinking mode.

## Step 1: Install Dependencies

In [2]:
# Install core dependencies for RAG system (compatible with Python 3.13)
!pip install -q chromadb sentence-transformers
!pip install -q PyMuPDF==1.25.1 pdfplumber==0.11.4
!pip install -q arabic-reshaper==3.0.0 python-bidi
!pip install -q rank-bm25 transformers torch
!pip install -q pydantic PyYAML
!pip install -q bitsandbytes accelerate huggingface-hub

# Install Flash Attention for faster inference (optional but recommended)
#!pip install -q flash-attn --no-build-isolation

#print('Dependencies installed successfully (Flash Attention enabled)')

In [3]:
# Verify GPU availability
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Sun Mar  8 18:44:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   29C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!kill -9 4450

## Step 2: Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted')

Mounted at /content/drive
Drive mounted


In [5]:
!pip install transformers --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 141.3 MB/s eta 0:00:0000:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Step 3: Setup HuggingFace Authentication

In [8]:
# Setup HuggingFace authentication
import os
from huggingface_hub import login

print("🔐 Setting up HuggingFace authentication...")
print("This will give you higher rate limits and faster model downloads.\n")

# Method 1: Environment variable (works in local Jupyter, VS Code, etc.)
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    print("✅ Using HF_TOKEN from environment variable")
    login(token=hf_token)
    print("🎉 Successfully authenticated with HuggingFace!\n")
else:
    print("❌ No HF_TOKEN found in environment variables")
    print("\n📋 Please set your HuggingFace token using one of these methods:")
    print("\n1️⃣  LOCAL/TERMINAL: Copy .env.example to .env and set HF_TOKEN=your_token")
    print("2️⃣  LOCAL/TERMINAL: Run: export HF_TOKEN=your_token_here")
    print("3️⃣  LOCAL/TERMINAL: Run: huggingface-cli login")
    print("4️⃣  GOOGLE COLAB: Runtime > Secrets > Add HF_TOKEN")
    print("5️⃣  MANUAL INPUT: Enter token below\n")
    
    # Try Colab secrets (only works in actual Colab UI)
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            login(token=hf_token)
            print("✅ Successfully authenticated with HuggingFace using Colab secret!\n")
        else:
            print("❌ No HF_TOKEN found in Colab secrets")
    except ImportError:
        print("ℹ️  Not running in Colab, skipping Colab secret check")
    except Exception as e:
        print(f"❌ Error accessing Colab secrets: {e}")
        print("💡 Make sure you're running this notebook in Google Colab UI (not VS Code or local Jupyter)")
        print("   To add secrets in Colab: Runtime > Secrets > Add HF_TOKEN\n")
    
    # Fallback: Manual token input
    try:
        user_input = input("🔑 Enter your HuggingFace token (or press Enter to skip): ").strip()
        if user_input:
            login(token=user_input)
            print("✅ Successfully authenticated with HuggingFace using manual input!\n")
        else:
            print("⚠️  Skipping authentication - you'll have lower rate limits but can still use the models")
    except KeyboardInterrupt:
        print("\n⚠️  Skipping authentication - you'll have lower rate limits but can still use the models")
    
    print("\n💡 Get your token from: https://huggingface.co/settings/tokens")

print("🔄 HuggingFace authentication setup complete!")

🔐 Setting up HuggingFace authentication...
This will give you higher rate limits and faster model downloads.

❌ No HF_TOKEN found in environment variables

📋 Please set your HuggingFace token using one of these methods:

1️⃣  LOCAL/TERMINAL: Copy .env.example to .env and set HF_TOKEN=your_token
2️⃣  LOCAL/TERMINAL: Run: export HF_TOKEN=your_token_here
3️⃣  LOCAL/TERMINAL: Run: huggingface-cli login
4️⃣  GOOGLE COLAB: Runtime > Secrets > Add HF_TOKEN
5️⃣  MANUAL INPUT: Enter token below



❌ Error accessing Colab secrets: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.
💡 Make sure you're running this notebook in Google Colab UI (not VS Code or local Jupyter)
   To add secrets in Colab: Runtime > Secrets > Add HF_TOKEN

✅ Successfully authenticated with HuggingFace using manual input!


💡 Get your token from: https://huggingface.co/settings/tokens
🔄 HuggingFace authentication setup complete!


## Step 4: Initialize RAG Pipeline with Qwen3.5

In [9]:
import sys, os, shutil, yaml

drive_rag_path = '/content/drive/MyDrive/QIAS26/qias_rag_thinking'
if os.path.exists(drive_rag_path):
    if os.path.exists('/content/qias_rag_thinking'):
        shutil.rmtree('/content/qias_rag_thinking')
    shutil.copytree(drive_rag_path, '/content/qias_rag_thinking')
    print('Copied RAG system from Drive')

config_path = '/content/qias_rag_thinking/config/rag_config.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    config['model']['name'] = 'Qwen3.5-9B-HF'
    config['model']['hf_model_name'] = 'Qwen/Qwen3.5-9B'
    config['model']['client_type'] = 'huggingface'
    config['model']['enable_thinking'] = True
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
    print('Config: Qwen/Qwen3.5-9B via HuggingFace (thinking enabled)')

sys.path.insert(0, '/content/qias_rag_thinking')
from src.rag_pipeline import RAGPipeline

print('\nLoading Qwen3.5 from HuggingFace...')
pipeline = RAGPipeline(config_path=config_path)
print('\nRAG Pipeline initialized')

Copied RAG system from Drive
Config: Qwen/Qwen3.5-9B via HuggingFace (thinking enabled)

Loading Qwen3.5 from HuggingFace...
Initializing RAG Pipeline...
Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

✅ Model loaded with eager attention
✓ Model loaded: Qwen/Qwen3.5-9B
No documents in vector store for BM25 index
RAG Pipeline initialized successfully

RAG Pipeline initialized


## Step 5: Build Knowledge Base from PDFs

In [10]:
pdf_directory = '/content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs'

import os
pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]
print(f'Found {len(pdf_files)} PDF files:')
for f in pdf_files:
    size_mb = os.path.getsize(os.path.join(pdf_directory, f)) / (1024*1024)
    print(f'  - {f} ({size_mb:.2f} MB)')

if pdf_files:
    pipeline.build_knowledge_base(pdf_directory)
    stats = pipeline.vector_store.get_collection_stats()
    print(f'\nKnowledge Base: {stats["total_documents"]} documents')
else:
    print(f'No PDFs found at {pdf_directory}')

Found 1 PDF files:
  - arabic_talkhis_fiqh_al_fraid.pdf (0.09 MB)

=== Building Knowledge Base ===
Found 1 PDF files in /content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs
Processed arabic_talkhis_fiqh_al_fraid.pdf: 65 chunks from 19 pages
Total documents created: 65
Generating embeddings for 65 documents...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Added batch 1: 65 documents
Total documents in collection: 65
Building BM25 index from vector store...
Built BM25 index with 65 documents
Knowledge base built with 65 document chunks

Knowledge Base: 65 documents


## Step 6: Test Single Question (QIAS 2026 format)

In [8]:
import json

test_question = 'مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟'
print(f'Question: {test_question}')
print(f'Model: {pipeline.qwen_client.model_name}\n')

result = pipeline.query(test_question, top_k=5)
raw = result.get('raw_output', '')
parsed = result['parsed_output']

# Thinking
print('='*80)
print('THINKING:')
print('='*80)
thinking = parsed.get('thinking', '')
if thinking:
    print(thinking[:2000])
else:
    print('No thinking captured')
    print(f'\nRaw (first 500 chars):\n{raw[:500]}')

# Structured output
print('\n' + '='*80)
print('PREDICTION OUTPUT (compared against ground truth):')
print('='*80)
structured = parsed.get('structured_output')
if structured and isinstance(structured, dict) and 'heirs' in structured:
    print(json.dumps(structured, ensure_ascii=False, indent=2))
    required = {'heirs', 'blocked', 'shares', 'tasil_stage', 'awl_or_radd', 'post_tasil'}
    present = required & set(structured.keys())
    missing = required - present
    print(f'\nSchema: {len(present)}/{len(required)} keys')
    if missing:
        print(f'Missing: {missing}')
else:
    print('Could not extract structured output')
    if parsed.get('answer_text'):
        print(f'\nAnswer text:\n{parsed["answer_text"][:500]}')

# Metrics
print('\n' + '='*80)
print('METRICS:')
print('='*80)
print(f'Parsing:    {parsed.get("parsing_success", False)}')
print(f'Validation: {parsed.get("validation_success", False)}')
print(f'Response:   {len(raw)} chars')
if parsed.get('error'):
    print(f'Error:      {parsed["error"]}')

Question: مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟
Model: Qwen/Qwen3.5-9B


=== Processing Question ===
مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟...
Retrieving documents...
Reranking documents...
Building prompt...
Generating response with Qwen3...
Generated 2228 chars in 223.7s
Parsing output...
✓ Parsing success: True
✓ Validation success: True
THINKING:
أهلاً بك. سأقوم بتحليل مسألة الميراث هذه خطوة بخطوة وفقاً لأحكام الفقه الإسلامي.

**الخطوة الأولى: تحديد الورثة ومن لا يرث**
الورثة المذكورون في السؤال هم:
1. الأم: وهي وارثة بالفرض.
2. الأب: وهو وارث بالتعصيب (عصبة).
3. البنت: وهي وارثة بالفرض.

لا يوجد محجوبون في هذه الحالة لأن الأم والأب والبنت لا يحجب بعضهم بعضاً في هذه التركيبة.

**الخطوة الثانية: تحديد المحجوبين من الورثة**
في هذه الحالة، لا يوجد محجوبون. فالأم والأب والبنت جميعهم يرثون.

**الخطوة الثالثة: تقدير الأنصبة الشرعية لأصحاب الفروض والعصبات**
- الأم: لها السدس (1/6) كالفرض.
- البنت: لها النصف (1/2) كالفرض.
- الأب: يأخذ الباقي بالتعصيب.

**الخطوة الرابعة: تأصيل 

## Step 7: Batch Evaluation on QIAS Dataset

In [10]:
## Step 8: Batch Evaluation on All Validation Dataset Files with Competition Metrics

import json
import os
from pathlib import Path
from src.model.evaluator import QIAS2026Evaluator

# Define paths
dev_data_dir = '/content/drive/MyDrive/QIAS26/data/dev'
output_base_dir = '/content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from pdf_only'
print(output_base_dir)
# Ensure output directory exists
os.makedirs(output_base_dir, exist_ok=True)

# Find all JSON files in the dev directory
dev_files = list(Path(dev_data_dir).glob('*.json'))
print(f'Found {len(dev_files)} validation dataset files:')
for f in dev_files:
    print(f'  - {f.name}')

# Process each validation file
for dev_file in dev_files:
    print(f'\n{"="*80}')
    print(f'Processing: {dev_file.name}')
    print(f'{"="*80}')

    try:
        # Load the dataset
        with open(dev_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)

        print(f'Loaded {len(dataset)} questions from {dev_file.name}')

        # Process questions in batches of 10 with intermediate saves
        batch_size = 10
        all_results = []
        output_data = []

        for batch_start in range(0, len(dataset), batch_size):
            batch_end = min(batch_start + batch_size, len(dataset))
            batch_dataset = dataset[batch_start:batch_end]

            print(f'Processing batch {batch_start//batch_size + 1}: questions {batch_start + 1}-{batch_end}...')

            # Process this batch
            batch_results = pipeline.batch_query(batch_dataset, save_results=False)
            all_results.extend(batch_results)

            # Prepare output data for current progress
            for i, (question_data, result) in enumerate(zip(batch_dataset, batch_results)):
                output_item = {
                    'question_id': batch_start + i + 1,
                    'question': question_data.get('question', ''),
                    'ground_truth': question_data.get('output', {}),
                    'model_prediction': result['parsed_output'].get('structured_output', {}),
                    'parsing_success': result['parsed_output'].get('parsing_success', False),
                    'validation_success': result['parsed_output'].get('validation_success', False),
                    'raw_response': result.get('raw_output', ''),
                    'thinking': result['parsed_output'].get('thinking', ''),
                    'processing_time': result.get('processing_time', 0)
                }
                output_data.append(output_item)
                print(f"question:\n{question_data.get('question', '')}")
                print(f"Raw_output:\n{result.get('raw_output', '')}")
            
            # Save intermediate results every batch
            output_filename = dev_file.stem + '_results.json'
            output_path = os.path.join(output_base_dir, output_filename)

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, ensure_ascii=False, indent=2)

            # Calculate and show statistics for current progress
            total_processed = len(output_data)
            parsing_success = sum(1 for r in output_data if r['parsing_success'])
            validation_success = sum(1 for r in output_data if r['validation_success'])

            print(f'📊 Progress saved: {total_processed}/{len(dataset)} questions')
            print(f'   Parsing success: {parsing_success}/{total_processed} ({parsing_success/total_processed*100:.1f}%)')
            print(f'   Validation success: {validation_success}/{total_processed} ({validation_success/total_processed*100:.1f}%)')
            print(f'✅ Intermediate results saved to: {output_path}')

        print(f'\n🎉 Completed processing {dev_file.name}')

        # Final statistics
        total_questions = len(output_data)
        parsing_success = sum(1 for r in output_data if r['parsing_success'])
        validation_success = sum(1 for r in output_data if r['validation_success'])

        print(f'📊 Final Statistics for {dev_file.name}:')
        print(f'   Total questions: {total_questions}')
        print(f'   Parsing success: {parsing_success}/{total_questions} ({parsing_success/total_questions*100:.1f}%)')
        print(f'   Validation success: {validation_success}/{total_questions} ({validation_success/total_questions*100:.1f}%)')

    except Exception as e:
        print(f'❌ Error processing {dev_file.name}: {str(e)}')
        continue

print(f'\n{"="*80}')
print('BATCH EVALUATION COMPLETE')
print(f'{"="*80}')
print(f'All results saved to: {output_base_dir}')
print('You can now copy these files back to your local workspace if needed.')

/content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from pdf_only
Found 2 validation dataset files:
  - qias2025_almawarith_part1.json
  - qias2025_almawarith_part61.json

Processing: qias2025_almawarith_part1.json
Loaded 100 questions from qias2025_almawarith_part1.json
Processing batch 1: questions 1-10...

=== Processing 10 Questions ===

[1/10]

=== Processing Question ===
مات وترك: خمسة أبناء ابن ابن و أم أم الأم و أربعة أبناء أخ لأب ما هو نصيب كل وريث؟...
Retrieving documents...
Reranking documents...
Building prompt...
Generating response with Qwen3...
Generated 2810 chars in 249.5s
Parsing output...
✓ Parsing success: True
✓ Validation success: False

[2/10]

=== Processing Question ===
مات وترك: أب الأب و أخوان شقيقان و ابن أخ شقيق و أربع بنات ابن و أربعة أبناء عم شقيق و خمسة أبناء اب...
Retrieving documents...
Reranking documents...
Building prompt...
Generating response with Qwen3...
Generated 2780 chars in 248.5s
Parsing output...
✓ Parsing success: True
✓ Validatio

## Step 9: QIAS 2026 Competition Evaluation Report

In [12]:
import json
from pathlib import Path
from collections import defaultdict
from src.model.evaluator import QIAS2026Evaluator

# Initialize evaluator
evaluator = QIAS2026Evaluator()

def find_result_files():
    """Robust file finding for Colab environment"""
    search_paths = [
        '/content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from pdf_only',  # space
        # '/content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from_pdf_only',  # underscore
        # '/content/drive/MyDrive/QIAS26/qias_rag_thinking/results',
        # '/content/drive/MyDrive/QIAS26/qias_rag_thinking/',
        # '/content/drive/MyDrive/QIAS26/results',
        # '/content/drive/MyDrive/QIAS26/',
        # '/content/drive/MyDrive/results',
        # '/content/drive/MyDrive/',
        # '/content/results',
        # '/content/',
        # './results',
        # '.'
    ]
    
    result_files = []
    
    for base_dir in search_paths:
        if Path(base_dir).exists():
            files = list(Path(base_dir).glob('*_results.json'))
            # Also check for specific files
            for specific_file in ['qias2025_almawarith_part1_results.json', 'qias2025_almawarith_part61_results.json']:
                specific_path = Path(base_dir) / specific_file
                if specific_path.exists() and specific_path not in files:
                    files.append(specific_path)
            
            if files:
                result_files.extend(files)
                print(f"✅ Found {len(files)} files in: {base_dir}")
                for f in files:
                    print(f"  📄 {f.name}")
                break  # Use first directory with results
    
    if not result_files:
        print("❌ No result files found in any location!")
        print("\n🔍 Searched in:")
        for path in search_paths:
            print(f"  {path}")
        return []
    
    return result_files

# Find result files
result_files = find_result_files()

if not result_files:
    print("\n💡 Troubleshooting tips:")
    print("1. Make sure Google Drive is mounted")
    print("2. Check the exact file names and locations")
    print("3. Run: !find /content/drive/MyDrive/ -name \"*results*.json\" -type f")
else:
    print(f"\n📊 Processing {len(result_files)} result files...")
    
    # Process each result file and generate competition evaluation
    all_predictions = []
    all_ground_truths = []
    all_individual_results = []

    for result_file in result_files:
        print(f'\n📊 Processing {result_file.name}...')
        
        with open(result_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        
        # Extract predictions and ground truths
        predictions = [r['model_prediction'] for r in results]
        ground_truths = [r['ground_truth'] for r in results]
        
        all_predictions.extend(predictions)
        all_ground_truths.extend(ground_truths)
        
        # Evaluate this dataset
        evaluation_report = evaluator.evaluate_dataset(predictions, ground_truths)
        
        # Add competition scores to results and save
        for i, result in enumerate(results):
            competition_scores = evaluation_report['individual_results'][i]
            result.update({
                'competition_overall_accuracy': competition_scores['overall_accuracy'],
                'competition_heirs_accuracy': competition_scores['heirs_accuracy'],
                'competition_blocked_accuracy': competition_scores['blocked_accuracy'],
                'competition_shares_accuracy': competition_scores['shares_accuracy'],
                'competition_awl_radd_accuracy': competition_scores['awl_radd_accuracy'],
                'competition_tasil_stage_accuracy': competition_scores['tasil_stage_accuracy'],
                'competition_post_tasil_accuracy': competition_scores['post_tasil_accuracy'],
                'competition_detailed_scores': competition_scores['detailed_scores']
            })
        
        # Save updated results
        with open(result_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        
        # Generate and display report for this file
        report = evaluator.generate_report(evaluation_report)
        print(f'\n📈 {result_file.stem.replace("_results", "")} Results:')
        print(f'   Overall Accuracy: {evaluation_report["aggregate_scores"]["overall_accuracy"]:.3f}')
        print(f'   Dataset Size: {evaluation_report["dataset_size"]} samples')
        
        # Store for overall report
        all_individual_results.extend(evaluation_report['individual_results'])

    # Generate overall competition report
    if all_predictions and all_ground_truths:
        print(f'\n🏆 Generating Overall QIAS 2026 Competition Report...')
        overall_evaluation = {
            'dataset_size': len(all_predictions),
            'individual_results': all_individual_results,
            'aggregate_scores': {},
            'component_averages': {},
            'score_distribution': {}
        }
        
        # Calculate overall statistics
        component_scores = defaultdict(list)
        for result in all_individual_results:
            for component in ['overall_accuracy', 'heirs_accuracy', 'blocked_accuracy',
                             'shares_accuracy', 'awl_radd_accuracy', 'tasil_stage_accuracy',
                             'post_tasil_accuracy']:
                component_scores[component].append(result[component])
        
        for component, scores in component_scores.items():
            overall_evaluation['component_averages'][component] = {
                'mean': sum(scores) / len(scores),
                'min': min(scores),
                'max': max(scores),
                'std': (sum((x - sum(scores)/len(scores))**2 for x in scores) / len(scores))**0.5
            }
            
            # Score distribution
            bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
            distribution = [0] * (len(bins) - 1)
            for score in scores:
                for j in range(len(bins) - 1):
                    if bins[j] <= score < bins[j + 1]:
                        distribution[j] += 1
                        break
                if score == 1.0:
                    distribution[-1] += 1
            overall_evaluation['score_distribution'][component] = {
                'bins': [f'{bins[i]:.1f}-{bins[i+1]:.1f}' for i in range(len(bins)-1)],
                'counts': distribution
            }
        
        overall_evaluation['aggregate_scores']['overall_accuracy'] = overall_evaluation['component_averages']['overall_accuracy']['mean']
        
        # Calculate pass rates
        for component in component_scores.keys():
            pass_rate = sum(1 for score in component_scores[component] if score >= 0.8) / len(component_scores[component])
            overall_evaluation['component_averages'][component]['pass_rate_80'] = pass_rate
        
        # Generate and save overall report
        output_base_dir = str(result_files[0].parent) if result_files else './'
        overall_report = evaluator.generate_report(overall_evaluation, f'{output_base_dir}/qias2026_competition_report.txt')
        print(f'\n{overall_report}')
        
        print(f'\n✅ Competition evaluation complete!')
        print(f'📄 Detailed report saved to: {output_base_dir}/qias2026_competition_report.txt')
        print(f'📊 Individual sample scores added to all result files')
    else:
        print('❌ No valid results found for evaluation')

✅ Found 2 files in: /content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from pdf_only
  📄 qias2025_almawarith_part1_results.json
  📄 qias2025_almawarith_part61_results.json

📊 Processing 2 result files...

📊 Processing qias2025_almawarith_part1_results.json...

📈 qias2025_almawarith_part1 Results:
   Overall Accuracy: 0.153
   Dataset Size: 100 samples

📊 Processing qias2025_almawarith_part61_results.json...

📈 qias2025_almawarith_part61 Results:
   Overall Accuracy: 0.149
   Dataset Size: 100 samples

🏆 Generating Overall QIAS 2026 Competition Report...
Evaluation report saved to: /content/drive/MyDrive/QIAS26/qias_rag_thinking/val_results_from pdf_only/qias2026_competition_report.txt

QIAS 2026 COMPETITION EVALUATION REPORT

Dataset Size: 200 samples

PRIMARY METRIC:
.1f

COMPONENT-WISE PERFORMANCE:
--------------------------------------------------
12s
15s

12s
15s

12s
15s

12s
15s

12s
15s

12s
15s

12s
15s

SCORE DISTRIBUTIONS:
--------------------------------------------